# Phase 4 — Day 16: OpenAIAPI

**Date:** 2026-04-24  
**Topic:** `chat.completions`, message roles, and generation controls

Today you will learn to:

- Build a chat completion request.
- Use `system`, `user`, and `assistant` roles.
- Control answers with `temperature`, `max_tokens`, and `top_p`.
- Write a reusable API helper.
- Avoid common request mistakes.


In [ ]:
# Setup
import os
import json
import random
from types import SimpleNamespace
from pprint import pprint

# Placeholder for exercises.
# Replace ___ with your answer.
___ = None

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

print("Setup complete.")
print("OpenAI SDK installed:", OpenAI is not None)
print("OPENAI_API_KEY found:", bool(os.getenv("OPENAI_API_KEY")))


In [ ]:
# Sample data
# This notebook has no external file dependency.

support_tickets = [
    {
        "id": 101,
        "text": "Siparişim 45 dakikadır gelmedi ve kurye haritada hareket etmiyor.",
        "true_category": "delivery_delay",
    },
    {
        "id": 102,
        "text": "Kartımdan iki kere ödeme çekilmiş görünüyor.",
        "true_category": "payment_issue",
    },
    {
        "id": 103,
        "text": "Burger soğuk geldi ve içecek eksikti.",
        "true_category": "food_quality",
    },
    {
        "id": 104,
        "text": "Adresimi güncellemek istiyorum ama uygulamada hata alıyorum.",
        "true_category": "app_issue",
    },
]

sample_messages = [
    {"role": "system", "content": "You are a concise support classifier."},
    {"role": "user", "content": support_tickets[0]["text"]},
]

pprint(support_tickets)
print("\nExample messages:")
pprint(sample_messages)


## 1. What `chat.completions` does

A chat completion request sends a list of messages to a model. The model reads the conversation and returns the next assistant message.

The core pattern is: choose a model, pass `messages`, then read `response.choices[0].message.content`.


In [ ]:
# A tiny mock client so the notebook runs without internet or API cost.
# It mimics client.chat.completions.create(...), but it is only for learning.

class MockCompletions:
    def create(self, model, messages, temperature=0.2, max_tokens=80, top_p=1.0, **kwargs):
        if not isinstance(messages, list):
            raise TypeError("messages must be a list of dictionaries")
        for message in messages:
            if "role" not in message or "content" not in message:
                raise ValueError("each message needs role and content")

        user_text = " ".join(m["content"] for m in messages if m["role"] == "user")
        system_text = " ".join(m["content"] for m in messages if m["role"] == "system")
        assistant_history = " ".join(m["content"] for m in messages if m["role"] == "assistant")

        text_lower = user_text.lower()

        if any(w in text_lower for w in ["gelmedi", "gec", "kurye", "dakika", "late"]):
            category = "delivery_delay"
        elif any(w in text_lower for w in ["kart", "ödeme", "para", "çekilmiş", "payment", "charged"]):
            category = "payment_issue"
        elif any(w in text_lower for w in ["soğuk", "eksik", "burger", "içecek", "cold"]):
            category = "food_quality"
        elif any(w in text_lower for w in ["uygulama", "hata", "adres", "app", "error"]):
            category = "app_issue"
        else:
            category = "other"

        styles = [
            f"category: {category}\\nreason: keyword match from the ticket",
            f"I would label this as {category}. The ticket gives a clear signal.",
            f"Best category: {category}. Confidence: medium to high.",
        ]

        idx = 0 if temperature < 0.4 else random.randint(0, len(styles) - 1)
        answer = styles[idx]

        if "json" in system_text.lower():
            answer = json.dumps({"category": category, "confidence": 0.82}, ensure_ascii=False)

        if assistant_history:
            answer += "\\nNote: I used the previous assistant message as context."

        # max_tokens is roughly simulated as a word limit.
        answer = " ".join(answer.split()[:max_tokens])

        return SimpleNamespace(
            choices=[SimpleNamespace(message=SimpleNamespace(role="assistant", content=answer))],
            usage=SimpleNamespace(
                prompt_tokens=sum(len(m["content"].split()) for m in messages),
                completion_tokens=len(answer.split()),
                total_tokens=sum(len(m["content"].split()) for m in messages) + len(answer.split()),
            ),
        )

class MockChat:
    def __init__(self):
        self.completions = MockCompletions()

class MockOpenAI:
    def __init__(self):
        self.chat = MockChat()

USE_REAL_API = bool(os.getenv("OPENAI_API_KEY")) and OpenAI is not None
client = OpenAI() if USE_REAL_API else MockOpenAI()

print("Using real OpenAI API:", USE_REAL_API)
print("Using mock client:", not USE_REAL_API)


In [ ]:
# Basic chat completion request

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=sample_messages,
    temperature=0.2,
    max_tokens=60,
    top_p=1.0,
)

print(response.choices[0].message.content)
print("\nUsage:")
print(vars(response.usage))


## 2. Message roles

Roles tell the model who said what.

`system` sets behavior. `user` asks for work. `assistant` can store earlier model replies, so the next request has conversation context.


In [ ]:
# Same user text, different system instruction

ticket = support_tickets[1]["text"]

friendly_messages = [
    {"role": "system", "content": "You are a friendly support assistant. Answer in one sentence."},
    {"role": "user", "content": ticket},
]

json_messages = [
    {"role": "system", "content": "You are a support classifier. Return JSON only."},
    {"role": "user", "content": ticket},
]

friendly_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=friendly_messages,
    temperature=0.2,
    max_tokens=60,
)

json_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=json_messages,
    temperature=0.2,
    max_tokens=60,
)

print("Friendly style:")
print(friendly_response.choices[0].message.content)

print("\nJSON style:")
print(json_response.choices[0].message.content)


In [ ]:
# Assistant history gives context to the next turn.

conversation = [
    {"role": "system", "content": "You classify support tickets. Be short."},
    {"role": "user", "content": "Kartımdan iki kere ödeme çekilmiş görünüyor."},
    {"role": "assistant", "content": "category: payment_issue"},
    {"role": "user", "content": "Bunu müşteriye nasıl kısa ve nazik açıklarsın?"},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=conversation,
    temperature=0.3,
    max_tokens=80,
)

print(response.choices[0].message.content)


## 3. `temperature`

`temperature` controls randomness. Low values are better for classification, extraction, and stable answers.

Higher values can help with brainstorming and varied wording. They can also make outputs less consistent.


In [ ]:
# Compare low and high temperature.
# The mock client makes low temperature stable and high temperature more varied.

messages = [
    {"role": "system", "content": "You are a concise support classifier."},
    {"role": "user", "content": support_tickets[2]["text"]},
]

for temp in [0.0, 0.2, 0.8, 1.0]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=temp,
        max_tokens=60,
    )
    print(f"temperature={temp}:")
    print(response.choices[0].message.content)
    print()


## 4. `max_tokens`

`max_tokens` limits how long the answer can be. Think of it as the output budget.

For labels, keep it small. For summaries or explanations, give the model more room.


In [ ]:
# Compare short and longer output budgets.

messages = [
    {"role": "system", "content": "You are a support classifier. Explain briefly."},
    {"role": "user", "content": support_tickets[0]["text"]},
]

for limit in [3, 8, 30]:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=0.2,
        max_tokens=limit,
    )
    print(f"max_tokens={limit}:")
    print(response.choices[0].message.content)
    print()


## 5. `top_p`

`top_p` is another randomness control. It limits token choices to the most likely group of options.

Usually tune either `temperature` or `top_p`, not both. For most projects, start with `temperature` and leave `top_p=1.0`.


In [ ]:
# The mock client accepts top_p so the API shape is clear.
# In real calls, top_p affects sampling. Here we print the request settings.

settings = [
    {"temperature": 0.2, "top_p": 1.0},
    {"temperature": 0.8, "top_p": 1.0},
    {"temperature": 0.8, "top_p": 0.5},
]

for setting in settings:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=sample_messages,
        max_tokens=60,
        **setting,
    )
    print(setting)
    print(response.choices[0].message.content)
    print()


## 6. A reusable helper function

Real projects should not repeat long API calls everywhere. Put the call in one function.

This makes your notebook cleaner and lets you change defaults in one place.


In [ ]:
def chat_complete(messages, model="gpt-4o-mini", temperature=0.2, max_tokens=80, top_p=1.0):
    # Small wrapper around chat.completions.create.
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p,
    )
    return response.choices[0].message.content

messages = [
    {"role": "system", "content": "You are a support classifier. Return JSON only."},
    {"role": "user", "content": support_tickets[3]["text"]},
]

answer = chat_complete(messages)
print(answer)


In [ ]:
# Use the helper on all sample tickets.

results = []

for ticket in support_tickets:
    messages = [
        {"role": "system", "content": "You are a support classifier. Return JSON only."},
        {"role": "user", "content": ticket["text"]},
    ]
    raw_answer = chat_complete(messages, temperature=0.2, max_tokens=40)
    results.append({
        "id": ticket["id"],
        "text": ticket["text"],
        "true_category": ticket["true_category"],
        "raw_answer": raw_answer,
    })

pprint(results)


## Tricky bits

Most API problems are simple request shape problems.

Check that `messages` is a list, each message has `role` and `content`, and your output length is not too tiny.


In [ ]:
# Mistake 1: messages is not a list.

bad_messages = {"role": "user", "content": "Hello"}

try:
    client.chat.completions.create(
        model="gpt-4o-mini",
        messages=bad_messages,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)


In [ ]:
# Mistake 2: missing role or content.

bad_messages = [
    {"role": "system", "content": "Classify support tickets."},
    {"text": "Kartımdan iki kere ödeme çekilmiş."},
]

try:
    client.chat.completions.create(
        model="gpt-4o-mini",
        messages=bad_messages,
    )
except Exception as error:
    print(type(error).__name__)
    print(error)


In [ ]:
# Mistake 3: max_tokens is too small.

messages = [
    {"role": "system", "content": "You are a helpful assistant. Explain the issue."},
    {"role": "user", "content": support_tickets[0]["text"]},
]

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages,
    temperature=0.2,
    max_tokens=2,
)

print("Too short output:")
print(response.choices[0].message.content)


## Trick questions

1. Should every request have a `system` message?

<details>
<summary>Answer</summary>
No. But it is useful when you want stable behavior, style, or constraints.
</details>

2. Why is low `temperature` better for classification?

<details>
<summary>Answer</summary>
Classification needs consistent labels. Low temperature reduces variation.
</details>

3. Does `max_tokens` limit the input or the output?

<details>
<summary>Answer</summary>
It limits the generated output. Your input also uses tokens, but that is part of the model context.
</details>

4. Should you tune `temperature` and `top_p` together?

<details>
<summary>Answer</summary>
Usually no. Start by tuning one. Most teams leave `top_p=1.0` and tune `temperature`.
</details>

5. Why include previous `assistant` messages?

<details>
<summary>Answer</summary>
They preserve conversation context. The model can see what it already answered.
</details>


## Exercises

Fill the `___` placeholders. Run each cell after editing it.


In [ ]:
# Exercise 1
# Create a user message dictionary for the first ticket.

message = {
    "role": ___,
    "content": ___,
}

assert message["role"] == "user"
assert message["content"] == support_tickets[0]["text"]
print("Exercise 1 passed.")


In [ ]:
# Exercise 2
# Create a system message that asks for JSON only.

system_message = {
    "role": ___,
    "content": ___,
}

assert system_message["role"] == "system"
assert "json" in system_message["content"].lower()
print("Exercise 2 passed.")


In [ ]:
# Exercise 3
# Build a messages list with system and user messages.

messages = [
    {"role": "system", "content": "You are a support classifier. Return JSON only."},
    {"role": ___, "content": support_tickets[1]["text"]},
]

assert isinstance(messages, list)
assert messages[1]["role"] == "user"
assert "Kartımdan" in messages[1]["content"]
print("Exercise 3 passed.")


In [ ]:
# Exercise 4
# Make a stable classification call.
# Use low temperature and a small output budget.

answer = chat_complete(
    messages=[
        {"role": "system", "content": "You are a support classifier. Return JSON only."},
        {"role": "user", "content": support_tickets[2]["text"]},
    ],
    temperature=___,
    max_tokens=___,
)

assert "food_quality" in answer
print(answer)
print("Exercise 4 passed.")


In [ ]:
# Exercise 5
# Read the assistant content from a raw response object.

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=sample_messages,
    temperature=0.2,
    max_tokens=40,
)

content = ___

assert isinstance(content, str)
assert len(content) > 0
print(content)
print("Exercise 5 passed.")


In [ ]:
# Exercise 6
# Classify every ticket with the helper.

predictions = []

for ticket in support_tickets:
    messages = [
        {"role": "system", "content": "You are a support classifier. Return JSON only."},
        {"role": "user", "content": ticket["text"]},
    ]
    prediction = ___
    predictions.append(prediction)

assert len(predictions) == len(support_tickets)
assert all(isinstance(p, str) for p in predictions)
print(predictions)
print("Exercise 6 passed.")


In [ ]:
# Exercise 7
# Choose the better setting for deterministic label extraction.
# Fill with 0.1 or 0.9.

best_temperature_for_labels = ___

assert best_temperature_for_labels == 0.1
print("Exercise 7 passed.")


## Exercise solutions

<details>
<summary>Exercise 1</summary>

```python
message = {
    "role": "user",
    "content": support_tickets[0]["text"],
}
```
</details>

<details>
<summary>Exercise 2</summary>

```python
system_message = {
    "role": "system",
    "content": "Return JSON only.",
}
```
</details>

<details>
<summary>Exercise 3</summary>

```python
messages = [
    {"role": "system", "content": "You are a support classifier. Return JSON only."},
    {"role": "user", "content": support_tickets[1]["text"]},
]
```
</details>

<details>
<summary>Exercise 4</summary>

```python
answer = chat_complete(
    messages=[
        {"role": "system", "content": "You are a support classifier. Return JSON only."},
        {"role": "user", "content": support_tickets[2]["text"]},
    ],
    temperature=0.2,
    max_tokens=40,
)
```
</details>

<details>
<summary>Exercise 5</summary>

```python
content = response.choices[0].message.content
```
</details>

<details>
<summary>Exercise 6</summary>

```python
prediction = chat_complete(messages, temperature=0.2, max_tokens=40)
```
</details>

<details>
<summary>Exercise 7</summary>

```python
best_temperature_for_labels = 0.1
```
</details>


## Cumulative review exercises

These review the last 10 days before today. They mix pandas, train/test split, pipelines, metrics, NLP, vectorization, and the complaint classification project.


In [ ]:
# Review setup data
import pandas as pd
import numpy as np

review_df = pd.DataFrame({
    "ticket_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "text": [
        "order is late",
        "payment failed",
        "food was cold",
        "app has error",
        "courier is late",
        "charged twice",
        "drink was missing",
        "app screen freezes",
    ],
    "category": ["delay", "payment", "quality", "app", "delay", "payment", "quality", "app"],
    "priority": [3, 2, 2, 1, 3, 2, 2, 1],
})

review_df


In [ ]:
# Review 1: pandas filtering
# Select only high priority tickets.

high_priority = review_df[___]

assert len(high_priority) == 2
assert set(high_priority["category"]) == {"delay"}
print("Review 1 passed.")
high_priority


In [ ]:
# Review 2: groupby aggregation
# Count tickets per category.

category_counts = review_df.groupby(___).size().sort_values(ascending=False)

assert category_counts.loc["delay"] == 2
assert category_counts.loc["payment"] == 2
print("Review 2 passed.")
category_counts


In [ ]:
# Review 3: train/test split
from sklearn.model_selection import train_test_split

X = review_df["text"]
y = review_df["category"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=___,
    random_state=42,
    stratify=y,
)

assert len(X_train) == 4
assert len(X_test) == 4
print("Review 3 passed.")


In [ ]:
# Review 4: TF-IDF vectorization
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(review_df["text"])

feature_names = vectorizer.get_feature_names_out()

assert ___ in feature_names
assert X_vec.shape[0] == len(review_df)
print("Review 4 passed.")
print(feature_names)


In [ ]:
# Review 5: pipeline
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe = Pipeline([
    ("tfidf", ___),
    ("model", LogisticRegression(max_iter=1000)),
])

assert "tfidf" in pipe.named_steps
assert "model" in pipe.named_steps
print("Review 5 passed.")


In [ ]:
# Review 6: fit a text classifier
# Use the whole tiny dataset only for practice.

pipe.fit(review_df["text"], review_df["category"])
pred = pipe.predict(["my order is very late"])[0]

assert pred in set(review_df["category"])
print("Prediction:", pred)
print("Review 6 passed.")


In [ ]:
# Review 7: classification metrics
from sklearn.metrics import accuracy_score

y_true = ["delay", "payment", "quality", "app"]
y_pred = ["delay", "payment", "delay", "app"]

acc = ___(y_true, y_pred)

assert round(acc, 2) == 0.75
print("Accuracy:", acc)
print("Review 7 passed.")


In [ ]:
# Review 8: confusion matrix
from sklearn.metrics import confusion_matrix

labels = ["app", "delay", "payment", "quality"]
cm = confusion_matrix(y_true, y_pred, labels=___)

assert cm.shape == (4, 4)
assert cm[0, 0] == 1
print(cm)
print("Review 8 passed.")


In [ ]:
# Review 9: simple keyword baseline
# Create a rule-based baseline for delay tickets.

def rule_based_delay_detector(text):
    text = text.lower()
    return ___ in text or "courier" in text

assert rule_based_delay_detector("order is late") is True
assert rule_based_delay_detector("payment failed") is False
print("Review 9 passed.")


In [ ]:
# Review 10: package predictions into a DataFrame

texts = ["order is late", "charged twice"]
predictions = pipe.predict(texts)

prediction_df = pd.DataFrame({
    "text": ___,
    "prediction": ___,
})

assert list(prediction_df.columns) == ["text", "prediction"]
assert len(prediction_df) == 2
print("Review 10 passed.")
prediction_df


## Cumulative review solutions

<details>
<summary>Review 1</summary>

```python
high_priority = review_df[review_df["priority"] == 3]
```
</details>

<details>
<summary>Review 2</summary>

```python
category_counts = review_df.groupby("category").size().sort_values(ascending=False)
```
</details>

<details>
<summary>Review 3</summary>

```python
test_size=0.5
```
</details>

<details>
<summary>Review 4</summary>

```python
assert "late" in feature_names
```
</details>

<details>
<summary>Review 5</summary>

```python
pipe = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("model", LogisticRegression(max_iter=1000)),
])
```
</details>

<details>
<summary>Review 7</summary>

```python
acc = accuracy_score(y_true, y_pred)
```
</details>

<details>
<summary>Review 8</summary>

```python
cm = confusion_matrix(y_true, y_pred, labels=labels)
```
</details>

<details>
<summary>Review 9</summary>

```python
return "late" in text or "courier" in text
```
</details>

<details>
<summary>Review 10</summary>

```python
prediction_df = pd.DataFrame({
    "text": texts,
    "prediction": predictions,
})
```
</details>


In [ ]:
# Cheat sheet

cheat_sheet = """
OPENAI CHAT COMPLETIONS QUICK REFERENCE

Basic shape:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are helpful."},
        {"role": "user", "content": "Classify this ticket."},
    ],
    temperature=0.2,
    max_tokens=80,
    top_p=1.0,
)

Read output:
response.choices[0].message.content

Roles:
system     = behavior and rules
user       = user request
assistant  = previous model answer or conversation memory

Settings:
temperature low  = stable, good for labels and extraction
temperature high = varied, good for brainstorming
max_tokens       = output length budget
top_p            = sampling control, usually leave at 1.0

Good defaults for classification:
temperature=0.0 or 0.2
max_tokens=40 to 100
top_p=1.0
"""

print(cheat_sheet)


## Footer

Next up: **Day 17 — PromptEngineeringPatterns**
